# deciax usage

deciax runs [Laya](https://github.com/NandhaKishorM/laya) decision models on JAX. You give it a
*state* (a message, a record, or a conversation) and a set of typed *questions*; it answers all of them
in one batched forward pass.

This notebook covers:

1. Loading a checkpoint
2. The three question types: `choice`, `score` and `noul`
3. State formats: string, dict and conversation
4. Question options: `labels`, `option_order` and structured criteria
5. Batching many states with `decide_batch`
6. Token budgets, the multilingual checkpoint, and throughput

## 1. Load a checkpoint

`deciax.load` downloads the pinned revision from the Hugging Face Hub (only the files the model reads)
and builds a jit engine. `dtype="auto"` is bf16 on TPU and fp32 elsewhere.

In [1]:
import json
import time

import jax

import deciax

print("deciax", deciax.__version__, "| backend:", jax.default_backend(), "|", jax.devices())

/home/junyi/deciax/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E1012 08:54:05.274736 1403203 hugepage_text.cc:344] RAW: File offset incorrectly aligned for file-backed THP: (600000 & ~ffffffffffe00000) = 0 != 1000 = (201000 & ~ffffffffffe00000)


deciax 0.1.0 | backend: tpu | [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


In [2]:
from deciax.models import MODELS

for name, ckpt in MODELS.items():
    print(f"{name:20} {ckpt.repo}@{ckpt.revision[:8]}")

laya                 convaiinnovations/laya@55cf4c4e
laya-multilingual    convaiinnovations/laya-multilingual@e4e9ddf2


In [3]:
t0 = time.perf_counter()
decider = deciax.load("laya")
print(f"loaded in {time.perf_counter() - t0:.1f}s")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1334.07it/s]

loaded in 2.6s


## 2. Ask questions

Each question has an id (any string or int), a `type`, `instructions`, and type-specific `criteria`:

| Type | `criteria` | Answer field |
|---|---|---|
| `choice` | dict of label → description, or a list of labels | `choice`: the winning label |
| `score` | list of level descriptions, lowest first | `score`: expected level index, from 0 to n-1 |
| `noul` | optional dict with `"true"` / `"false"` descriptions | `noul`: probability of true |

The first call compiles the engine for this input shape, so it is slower than later calls.

In [4]:
questions = {
    "intent": {
        "type": "choice",
        "instructions": "What does the user want?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "cancel": "ending the plan",
            "upgrade": "buying more",
            "other": "everything else",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this?",
        "criteria": ["not urgent", "soon", "blocking"],
    },
    "angry": {"type": "noul", "instructions": "Is the user angry?"},
}

response = decider.decide(
    "I was charged twice for my subscription this month, please fix it ASAP.", questions
)
print(json.dumps(response, indent=2))

{
  "model": "laya-rl-agent",
  "answers": {
    "intent": {
      "type": "choice",
      "choice": "billing",
      "probabilities": {
        "billing": 0.979,
        "cancel": 0.0088,
        "upgrade": 0.0052,
        "other": 0.007
      },
      "confidence": 0.9101,
      "answer_confidence": 0.979,
      "action": {
        "act_probability": 1.0
      }
    },
    "urgency": {
      "type": "score",
      "score": 1.7663,
      "legend": {
        "0": "not urgent",
        "1": "soon",
        "2": "blocking"
      },
      "probabilities": {
        "0": 0.0395,
        "1": 0.1547,
        "2": 0.8058
      },
      "confidence": 0.4626,
      "answer_confidence": 0.8058,
      "action": {
        "act_probability": 1.0
      }
    },
    "angry": {
      "type": "noul",
      "noul": 0.7822,
      "confidence": 0.7822,
      "answer_confidence": 0.7822,
      "action": {
        "act_probability": 1.0
      }
    }
  },
  "usage": {
    "input_tokens": 146,
    "output_t

The response has the same shape as Laya's `system_one` response. Every answer also carries:

- `confidence` / `answer_confidence`: how sure the model is of its answer
- `action.act_probability`: whether the model thinks the answer warrants acting on

A small helper makes the rest of the notebook easier to read:

In [5]:
def summarize(response):
    for qid, a in response["answers"].items():
        value = a[a["type"]]
        if isinstance(value, float):
            value = round(value, 3)
        print(
            f"{qid:12} {a['type']:7} {value!s:12} "
            f"confidence={a['confidence']:.2f} act={a['action']['act_probability']:.2f}"
        )
    print("usage:", response["usage"])


summarize(response)

intent       choice  billing      confidence=0.91 act=1.00
urgency      score   1.766        confidence=0.46 act=1.00
angry        noul    0.782        confidence=0.78 act=1.00
usage: {'input_tokens': 146, 'output_tokens': 0, 'state_tokens': 16, 'state_tokens_dropped': 0, 'truncated': False, 'truncated_questions': []}


In [6]:
summarize(decider.decide("Hey, could you send me last month's invoice when you get a chance?", questions))

intent       choice  billing      confidence=0.92 act=1.00
urgency      score   0.677        confidence=0.07 act=1.00
angry        noul    0.0          confidence=1.00 act=1.00
usage: {'input_tokens': 146, 'output_tokens': 0, 'state_tokens': 16, 'state_tokens_dropped': 0, 'truncated': False, 'truncated_questions': []}


## 3. State formats

The state can be:

- a **string**: a message or document
- a **dict**: a structured record, serialised as JSON
- a **list**: a conversation. When it is longer than the token budget, the oldest turns are dropped
  first.

In [7]:
record = {
    "user": "Cancel my plan, I'm moving to a competitor.",
    "plan": "pro",
    "months_subscribed": 14,
}
summarize(decider.decide(record, questions))

intent       choice  cancel       confidence=1.00 act=1.00
urgency      score   1.845        confidence=0.60 act=1.00
angry        noul    0.728        confidence=0.73 act=1.00
usage: {'input_tokens': 185, 'output_tokens': 0, 'state_tokens': 29, 'state_tokens_dropped': 0, 'truncated': False, 'truncated_questions': []}


In [8]:
conversation = [
    {"role": "user", "content": "My router keeps dropping the connection every few minutes."},
    {"role": "assistant", "content": "Have you tried restarting it?"},
    {"role": "user", "content": "Three times already. This is the third call this week and I'm fed up."},
]
summarize(decider.decide(conversation, questions))

intent       choice  cancel       confidence=0.04 act=1.00
urgency      score   1.869        confidence=0.65 act=1.00
angry        noul    0.684        confidence=0.68 act=1.00
usage: {'input_tokens': 305, 'output_tokens': 0, 'state_tokens': 69, 'state_tokens_dropped': 0, 'truncated': False, 'truncated_questions': []}


## 4. Question options

- `labels` (noul only) renames the false/true options the model sees.
- `option_order` presents the options in a different order. The answer keys stay the same.
- `instructions` and criteria can be structured values. They are rendered as single-line JSON.

In [9]:
review = "The package arrived two weeks late and the box was crushed."
summarize(
    decider.decide(
        review,
        {
            "complaint": {
                "type": "noul",
                "instructions": {"task": "Is this a complaint?", "scope": "shipping"},
                "criteria": {"true": "the customer is unhappy", "false": "neutral or positive"},
                "labels": {"false": "no", "true": "yes"},
            },
            "department": {
                "type": "choice",
                "instructions": "Which department handles this?",
                "criteria": ["shipping", "billing", "product", "support"],
                "option_order": [2, 0, 3, 1],
            },
            "severity": {
                "type": "score",
                "instructions": "How severe is the problem?",
                "criteria": [
                    {"level": "minor"},
                    {"level": "moderate", "examples": ["late"]},
                    {"level": "severe"},
                ],
            },
        },
    )
)

complaint    noul    0.89         confidence=0.89 act=1.00
department   choice  shipping     confidence=0.12 act=1.00
severity     score   0.993        confidence=0.55 act=1.00
usage: {'input_tokens': 143, 'output_tokens': 0, 'state_tokens': 12, 'state_tokens_dropped': 0, 'truncated': False, 'truncated_questions': []}


Invalid questions raise `ValueError` with a message saying what to fix (the HTTP server returns 422):

In [10]:
try:
    decider.decide("hi", {"q": {"type": "score", "instructions": "How?", "criteria": {"low": "..."}}})
except ValueError as e:
    print("ValueError:", e)

ValueError: question 'q': a score question takes 'criteria' as a list of level descriptions, index 0 first


## 5. Batching

`decide_batch` asks the same questions about many states. Rows from all states share forward passes,
which is much faster than calling `decide` in a loop.

In [11]:
tickets = [
    "I was charged twice this month, please refund one of them.",
    "How do I add three more seats to our team plan?",
    "Please close my account, I no longer need it.",
    "Your app crashed and deleted my notes. Unacceptable!",
    "What are your opening hours?",
]
sentiment_questions = {
    "intent": questions["intent"],
    "angry": questions["angry"],
}

for ticket, r in zip(tickets, decider.decide_batch(tickets, sentiment_questions)):
    a = r["answers"]
    print(f"{a['intent']['choice']:8} angry={a['angry']['noul']:.2f}  {ticket}")

billing  angry=0.40  I was charged twice this month, please refund one of them.
upgrade  angry=0.00  How do I add three more seats to our team plan?
cancel   angry=0.54  Please close my account, I no longer need it.
cancel   angry=0.87  Your app crashed and deleted my notes. Unacceptable!
billing  angry=0.00  What are your opening hours?


## 6. Token budgets

`max_len` caps the whole sequence and `head_max_len` caps the question part. Both default to the
checkpoint's settings. Long states are truncated to fit; `usage.input_tokens` shows what was used.

In [12]:
long_state = "The router keeps dropping the connection. " * 200
for max_len in (None, 256):
    r = decider.decide(long_state, {"angry": questions["angry"]}, max_len=max_len)
    print(f"max_len={max_len!s:5} input_tokens={r['usage']['input_tokens']}")

max_len=None  input_tokens=512


max_len=256   input_tokens=256


## 7. Multilingual checkpoint

`laya-multilingual` handles non-English states. There is no automatic language routing: pick the
checkpoint yourself.

In [13]:
multi = deciax.load("laya-multilingual")
for text in [
    "Me cobraron dos veces este mes, por favor arréglenlo.",
    "这个月扣了我两次费，请尽快处理！",
    "Ich möchte mein Abo kündigen.",
]:
    a = multi.decide(text, questions)["answers"]
    print(f"{a['intent']['choice']:8} urgency={a['urgency']['score']:.2f} angry={a['angry']['noul']:.2f}  {text}")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2416.91it/s]

billing  urgency=1.69 angry=0.33  Me cobraron dos veces este mes, por favor arréglenlo.
billing  urgency=1.81 angry=0.03  这个月扣了我两次费，请尽快处理！
cancel   urgency=1.68 angry=0.00  Ich möchte mein Abo kündigen.


## 8. Throughput

The engine pads inputs to fixed shape buckets, so repeated calls with similar shapes reuse compiled
programs. Every (state, question) pair is one row of the batch, so cost scales with the number of
questions answered. Warm up once, then time it:

In [14]:
batch = tickets * 20  # 100 states x 2 questions = 200 rows
decider.decide_batch(batch, sentiment_questions)  # warm-up / compile

t0 = time.perf_counter()
n_iter = 5
for _ in range(n_iter):
    decider.decide_batch(batch, sentiment_questions)
dt = (time.perf_counter() - t0) / n_iter
n_questions = len(batch) * len(sentiment_questions)
print(f"{len(batch)} states x {len(sentiment_questions)} questions = {n_questions} questions in {dt * 1000:.1f} ms")
print(f"{dt / n_questions * 1e6:.0f} µs per question -> {n_questions / dt:.0f} questions/s")

100 states x 2 questions = 200 questions in 71.3 ms
356 µs per question -> 2806 questions/s


## Next steps

- Serve the same API over HTTP with `python -m deciax.serve` (see the README); it speaks the
  `/v1/systemone` wire protocol of `laya-serve`.
- `deciax.load(..., impl="pallas")` uses the Pallas flash-attention kernel instead of XLA attention.